# 🚀 Welcome to Your ADK Adventure - Tools & Memory! 🚀

Welcome, Agent Architect! This notebook is your guide to giving your AI agents two essential superpowers: custom tools and conversational memory.

By the end of this adventure, you will be able to:

- **Build a Foundational Agent**: Create a simple but effective AI agent from scratch using the Google Agent Development Kit (ADK).

- **Grant New Skills with Custom Tools**: Teach an agent to perform new tasks by connecting it to external APIs, like a real-time weather service.

- **Create a Team of Agents**: Assemble a multi-agent system where a primary agent can delegate specialized tasks to other agents.

- **Master Conversational Memory**: Understand the critical role of Sessions in enabling agents to remember previous interactions, handle feedback, and carry on a coherent conversation.


Let's get this adventure started!

## Author

HI, I'm Qingyue (Annie) Wang, a developer advocate and AI engineer at **Google**, passionate about helping developers build with AI and cloud technologies :)


If you have questions with this notebook, contact me on [LinkedIn](https://www.linkedin.com/in/anniewangtech/) , [X](https://twitter.com/anniewangtech) or email anniewangtech0510@Gmail.com


```
  (\__/)
  (•ㅅ•)
  /づ  📚      Enjoy learning AI Agents :)
```


-------------
### 🎁 🛑 Important Prerequisite: Setup Your Environment! 🛑 🎁
-----------------------------------------------------------------------------

👉 **Get Your API Key HERE**: https://codelabs.developers.google.com/onramp/instructions#1

 -----------------------------------------------------------------------------

```
 ⬆️  ⬆️  ⬆️  ⬆️  ⬆️  ⬆️  ⬆️  ⬆️  ⬆️  ⬆️  ⬆️  ⬆️  ⬆️  ⬆️  ⬆️
   /\_/\     /\_/\     /\_/\      /\_/\       /\_/\
  ( ^_^ )   ( -.- )   ( >_< )   ( =^.^= )    ( o_o )             
```


## Part 0: Setup & Authentication 🔑

First things first, let's get all our tools ready. This step installs the necessary libraries and securely configures your Google API key so your agents can access the power of Gemini.

In [ ]:
!pip install google-adk google-generativeai -q

# --- Import all necessary libraries for our entire adventure ---
import os
import re
import asyncio
from IPython.display import display, Markdown
import google as genai
from google.adk.agents import Agent
from google.adk.tools import google_search
from google.adk.runners import Runner
from google.adk.sessions import session
from google.adk.sessions import InMemorySessionService, Session
from google.genai.types import Content, Part
from getpass import getpass

print("✅ All libraries are ready to go!")

✅ All libraries are ready to go!


In [ ]:
# --- Securely Configure Your API Key ---
import os
import random

# Prompt the user for their API key securely
#api_key = getpass('Enter your Google API Key: ')

#Direct way to ADD Api-key
API_KEYS = ["",""

            ]

# Configure the generative AI library with the provided key
#genai.configure(api_key=api_key)

# Select a random API key
api_key = random.choice(API_KEYS)

# Set the API key as an environment variable for ADK to use
os.environ['GOOGLE_API_KEY'] = api_key

#Client = genai.Client(api_key=api_key)

print(f"✅ Random API Key selected: {api_key[:10]}********")
print("✅ API Key configured successfully! Let the fun begin.")

✅ Random API Key selected: AIzaSyC0M8********
✅ API Key configured successfully! Let the fun begin.


---
## Part 1: Your First Agent - The Day Trip Genie 🧞

Meet your first creation! The `day_trip_agent` is a simple but powerful assistant. We're making it a little smarter by teaching it to understand **budget constraints**.

* **Agent**: The brain of the operation, defined by its instructions, tools, and the AI model it uses.
* **Session**: The conversation history. For this simple agent, it's just a container for a single request-response.
* **Runner**: The engine that connects the `Agent` and the `Session` to process your request and get a response.

```
+--------------------------------------------------+
|         Spontaneous Day Trip Agent 🤖            |
|--------------------------------------------------|
|  Model: gemini-2.5-flash                         |
|  Description:                                    |
|   Generates full-day trip itineraries based on   |
|   mood, interests, and budget                    |
|--------------------------------------------------|
|  🔧 Tools:                                       |
|   - Google Search                                |
|--------------------------------------------------|
|  🧠 Capabilities:                                |
|   - Budget Awareness (cheap / splurge)           |
|   - Mood Matching (adventurous, relaxing, etc.)  |
|   - Real-Time Info (hours, events)               |
|   - Morning / Afternoon / Evening plan           |
+--------------------------------------------------+

            ▲
            |
    +------------------+
    |   User Input     |
    |------------------|
    |  Mood            |
    |  Interests       |
    |  Budget          |
    +------------------+

            |
            ▼

+--------------------------------------------------+
|             Output: Markdown Itinerary           |
|--------------------------------------------------|
| - Time blocks (Morning / Afternoon / Evening)    |
| - Venue names with links and hours               |
| - Budget-matching activities                     |
+--------------------------------------------------+
```


In [ ]:
# --- Agent Definition ---

def create_day_trip_agent():
    """Create the Spontaneous Day Trip Generator agent"""
    return Agent(
        name="day_trip_agent",
        model="gemini-2.5-flash",
        description="Agent specialized in generating spontaneous full-day itineraries based on mood, interests, and budget.",
        instruction="""
        You are the "Spontaneous Day Trip" Generator 🚗 - a specialized AI assistant that creates engaging full-day itineraries.

        Your Mission:
        Transform a simple mood or interest into a complete day-trip adventure with real-time details, while respecting a budget.

        Guidelines:
        1. **Budget-Aware**: Pay close attention to budget hints like 'cheap', 'affordable', or 'splurge'. Use Google Search to find activities (free museums, parks, paid attractions) that match the user's budget.
        2. **Full-Day Structure**: Create morning, afternoon, and evening activities.
        3. **Real-Time Focus**: Search for current operating hours and special events.
        4. **Mood Matching**: Align suggestions with the requested mood (adventurous, relaxing, artsy, etc.).

        RETURN itinerary in MARKDOWN FORMAT with clear time blocks and specific venue names.
        """,
        tools=[google_search]
    )

day_trip_agent = create_day_trip_agent()
print(f"🧞 Agent '{day_trip_agent.name}' is created and ready for adventure!")

🧞 Agent 'day_trip_agent' is created and ready for adventure!


In [ ]:
# --- A Helper Function to Run Our Agents ---
# We'll use this function throughout the notebook to make running queries easy.

async def run_agent_query(agent: Agent, query: str, session: Session, user_id: str, is_router: bool = False):
    """Initializes a runner and executes a query for a given agent and session."""
    print(f"\n🚀 Running query for agent: '{agent.name}' in session: '{session.id}'...")

    runner = Runner(
        agent=agent,
        session_service=session_service,
        app_name=agent.name
    )

    final_response = ""
    try:
        async for event in runner.run_async(
            user_id=user_id,
            session_id=session.id,
            new_message=Content(parts=[Part(text=query)], role="user")
        ):
            if not is_router:
                # Let's see what the agent is thinking!
                print(f"EVENT: {event}")
            if event.is_final_response():
                final_response = event.content.parts[0].text
    except Exception as e:
        final_response = f"An error occurred: {e}"

    if not is_router:
     print("\n" + "-"*50)
     print("✅ Final Response:")
     display(Markdown(final_response))
     print("-"*50 + "\n")

    return final_response

# --- Initialize our Session Service ---
# This one service will manage all the different sessions in our notebook.
session_service = InMemorySessionService()
my_user_id = "adk_adventurer_001"

In [ ]:
# --- Let's test the Day Trip Genie! ---

#Example:1

async def run_day_trip_genie():
    # Create a new, single-use session for this query
    day_trip_session = await session_service.create_session(
        app_name=day_trip_agent.name,
        user_id=my_user_id
    )

    # Note the new budget constraint in the query!
    query = "i am having a randomm plain to go to pamba from mysure  and i also want experience the western gahts and i am using public transport please help mw with it "
    #query = "Plan a relaxing and artsy day trip near Sunnyvale, CA. Keep it affordable!"
    print(f"🗣️ User Query: '{query}'")

    await run_agent_query(day_trip_agent, query, day_trip_session, my_user_id)

await run_day_trip_genie()

🗣️ User Query: 'i am having a randomm plain to go to pamba from mysure  and i also want experience the western gahts and i am using public transport please help mw with it '

🚀 Running query for agent: 'day_trip_agent' in session: '0274fc87-b447-4bbc-bfe6-ee1cc442392c'...
EVENT: model_version='gemini-2.5-flash' content=Content(
  parts=[
    Part(
      text="""Embarking on a spontaneous day trip from Mysuru to Pamba via the scenic Western Ghats using public transport is an adventurous journey that focuses on the experience of the travel itself. Given the significant distance, this will be a full day of travel, with the Western Ghats experience primarily enjoyed during the daylight hours of your bus journey into Kerala.

Here's a suggested itinerary:

## Spontaneous Western Ghats Odyssey: Mysuru to Pamba

**Budget:** Affordable (Public Bus Transport)
**Mood:** Adventurous, Scenic, Immersive Travel

---

### **Morning: Traversing the Western Ghats into Wayanad**

*   **6:00 AM - 9:00 AM

Embarking on a spontaneous day trip from Mysuru to Pamba via the scenic Western Ghats using public transport is an adventurous journey that focuses on the experience of the travel itself. Given the significant distance, this will be a full day of travel, with the Western Ghats experience primarily enjoyed during the daylight hours of your bus journey into Kerala.

Here's a suggested itinerary:

## Spontaneous Western Ghats Odyssey: Mysuru to Pamba

**Budget:** Affordable (Public Bus Transport)
**Mood:** Adventurous, Scenic, Immersive Travel

---

### **Morning: Traversing the Western Ghats into Wayanad**

*   **6:00 AM - 9:00 AM: Depart Mysuru to Sulthan Bathery (Wayanad, Kerala)**
    *   Head to the **Mysuru KSRTC Bus Stand**. Look for a Karnataka State Road Transport Corporation (KSRTC) or Kerala State Road Transport Corporation (KSRTC Kerala) bus heading towards **Sulthan Bathery** or **Kalpetta** in Wayanad, Kerala. There are frequent buses available from early morning.
    *   The journey typically takes around 2 to 3 hours.
    *   **Experience the Western Ghats:** This part of the journey is highly scenic as the bus will pass through the **Bandipur Tiger Reserve** (Karnataka) and **Mudumalai Wildlife Sanctuary** (Tamil Nadu), leading into the **Wayanad Ghats** of Kerala. Keep an eye out for wildlife and lush greenery. The winding roads and hairpin bends offer quintessential Western Ghats views. Traffic through Bandipur forest is allowed from 6 AM to 9 PM, so an early start ensures daylight views.
    *   **Estimated Fare:** ₹150 - ₹250 (Non-AC seater).

*   **9:00 AM - 10:00 AM: Breakfast & Refresh in Sulthan Bathery**
    *   Upon arrival at **Sulthan Bathery Bus Stand**, take a refreshing break. You'll find numerous local eateries (dhabas/restaurants) near the bus stand serving traditional Kerala breakfast items like Appam, Puttu, or Dosa with various curries. Enjoy some local filter coffee or tea.

### **Afternoon: Journey Deeper into Kerala**

*   **10:00 AM - 9:00 PM: Sulthan Bathery to Pathanamthitta**
    *   From Sulthan Bathery, board a **KSRTC (Kerala) bus** bound for **Pathanamthitta**. This will be the longest leg of your journey, taking approximately 10 to 11 hours.
    *   The route will take you through diverse landscapes of Kerala, from the hills of Wayanad down to the central plains. While not as intensely "Ghat" focused as the first leg, you'll still experience the changing topography and lush scenery characteristic of Kerala.
    *   **Lunch:** The bus will likely make a stop for lunch at a highway restaurant (known as 'hotel' in Kerala). Keep it simple and try a local 'meals' (rice with various curries and side dishes).
    *   **Estimated Fare:** ₹500 - ₹700 (depending on bus type).

### **Evening: Arrival in Pamba**

*   **9:00 PM - 11:30 PM: Pathanamthitta to Pamba**
    *   Upon reaching **Pathanamthitta KSRTC Bus Stand**, you will find frequent **KSRTC (Kerala) buses** heading to **Pamba**. Pathanamthitta is a major hub for Pamba-bound services.
    *   The final leg of the journey is relatively short, typically taking about 1 to 2 hours.
    *   **Estimated Fare:** ₹100 - ₹150.

*   **11:30 PM: Arrive in Pamba**
    *   You will arrive at **Pamba Bus Stand**, which serves as the base for pilgrims. Note that Pamba is primarily a pilgrimage destination.

**Important Notes for a Spontaneous Trip:**

*   **Bus Tickets:** For longer inter-state journeys, it's advisable to check online booking platforms (like redBus or Paytm) or the official KSRTC (Karnataka and Kerala) websites for schedules and to book tickets in advance, especially for the Mysuru to Wayanad leg and the Wayanad to Pathanamthitta leg. While "unplanned" is the mood, having confirmed long-haul tickets saves a lot of hassle.
*   **Flexibility:** Bus timings can be subject to delays due to traffic or road conditions, especially in Ghat sections. Stay flexible with your schedule.
*   **Food & Water:** Carry snacks and water for the long journey. You will have stops, but it's always good to have provisions.
*   **Accessibility in Pamba:** Pamba is mainly a transit point for pilgrims to Sabarimala. If you plan to stay, arrange accommodation beforehand, as options might be limited or geared towards pilgrims.
*   **"Random, Unplanned":** While the itinerary provides a structure, embrace the spontaneity of bus travel in India. Engage with fellow passengers, observe the local life flashing by, and enjoy the ever-changing vistas of the Western Ghats.

--------------------------------------------------



In [ ]:
# --- Let's test the Day Trip Genie! ---

#Example:2

async def run_day_trip_genie():
    # Create a new, single-use session for this query
    day_trip_session = await session_service.create_session(
        app_name=day_trip_agent.name,
        user_id=my_user_id
    )

    # Note the new budget constraint in the query!
    query = "Create a walking tour in Berkeley that combines coffee shops, street murals, and live music spots under $50."
    #query = "Plan a relaxing and artsy day trip near Sunnyvale, CA. Keep it affordable!"
    print(f"🗣️ User Query: '{query}'")

    await run_agent_query(day_trip_agent, query, day_trip_session, my_user_id)

await run_day_trip_genie()

🗣️ User Query: 'Create a walking tour in Berkeley that combines coffee shops, street murals, and live music spots under $50.'

🚀 Running query for agent: 'day_trip_agent' in session: 'bbf5def7-ac81-499f-b136-6ffc2810a6f2'...
EVENT: model_version='gemini-2.5-flash' content=Content(
  parts=[
    Part(
      text="""Here's a spontaneous walking tour in Berkeley designed to immerse you in its vibrant coffee scene, impressive street art, and lively music, all while keeping a budget under $50 for today, Thursday, March 5, 2026!

---

## Spontaneous Berkeley Day Trip: Coffee, Murals & Live Music

**Budget Goal:** Under $50 (requires mindful spending on food/drinks and securing the lower-priced concert ticket)

---

### **Morning (9:00 AM - 12:00 PM): Downtown Brews & Urban Canvases**

*   **9:00 AM - Coffee Kick-off at 1951 Coffee Company (Approx. $5-7)**
    Start your day with an ethically sourced and highly-rated coffee at **1951 Coffee Company** (2416 Channing Way, Berkeley, CA 94704). T

Here's a spontaneous walking tour in Berkeley designed to immerse you in its vibrant coffee scene, impressive street art, and lively music, all while keeping a budget under $50 for today, Thursday, March 5, 2026!

---

## Spontaneous Berkeley Day Trip: Coffee, Murals & Live Music

**Budget Goal:** Under $50 (requires mindful spending on food/drinks and securing the lower-priced concert ticket)

---

### **Morning (9:00 AM - 12:00 PM): Downtown Brews & Urban Canvases**

*   **9:00 AM - Coffee Kick-off at 1951 Coffee Company (Approx. $5-7)**
    Start your day with an ethically sourced and highly-rated coffee at **1951 Coffee Company** (2416 Channing Way, Berkeley, CA 94704). This spot is known for its excellent coffee and baked goods, and it's located conveniently near downtown. They typically close around 5:00 PM, so it's a great morning stop.
*   **10:00 AM - Downtown Berkeley Mural Exploration (Free)**
    After your coffee, begin your mural hunt in Downtown Berkeley. The area is described as an "open-air gallery" with dynamic artworks reflecting the city's culture and history. Wander around **Center Street, Shattuck Avenue, and Addison Street**. Keep an eye out for larger pieces on building sides and smaller "hidden gems" in alleys. Many murals showcase local artists and community themes, so take your time to appreciate the intricate details and vibrant colors.

### **Afternoon (12:00 PM - 5:00 PM): South Berkeley Murals & Campus Vibes**

*   **12:30 PM - Casual Lunch (Approx. $12-15)**
    As you make your way south from downtown along Shattuck Avenue, grab an affordable and delicious lunch from one of the many casual eateries or delis. Look for spots offering sandwiches, tacos, or other quick bites to stay within budget.
*   **1:30 PM - South Shattuck Mural Walk (Free)**
    Continue your artistic journey heading further south on Shattuck Avenue. You'll encounter more significant murals in this area. Be sure to stop at:
    *   **The Starry Plough Pub** (3101 Shattuck Ave): The wall facing Shattuck sports several murals celebrating music and struggle.
    *   **La Peña Cultural Center** (3105 Shattuck Ave): Just to the south of Starry Plough, this center features an "amazing, partially-three-dimensional mural" depicting a musician and historical figures.
    *   **Bonus Murals:** Keep an eye out as you walk towards the Ashby BART area, as there are mentions of murals near the intersection of Essex and Shattuck (3033 Shattuck) and a Yoda mural further south on Alcatraz Avenue (1601 Alcatraz).
*   **3:30 PM - Afternoon Pick-me-up at Caffè Strada (Approx. $5-8)**
    Head towards the UC Berkeley campus area for a classic Berkeley experience at **Caffè Strada** (2300 College Ave, Berkeley, CA 94704). This iconic spot is beloved by students and locals for its bustling atmosphere and ample outdoor seating, perfect for people-watching and soaking in the campus vibe. They are open until midnight.

### **Evening (7:00 PM - Late): Live Music in Downtown Berkeley**

*   **7:00 PM - Pre-Show Bites/Relax**
    Before the show, you can grab a light, budget-friendly dinner or snack in Downtown Berkeley, perhaps a slice of pizza or some street food, depending on what catches your eye.
*   **8:00 PM - Live Music at Cornerstone (Ticket Approx. $24.69 - $38.00)**
    Cap off your day with live music at **Cornerstone** (2367 Shattuck Ave, Berkeley, CA 94704). Tonight, **Thursday, March 5, 2026**, the band **Wheel** is performing, with tickets starting from $24.69. Cornerstone is a popular music venue in the heart of Downtown Berkeley, offering an authentic local music experience.

---

**Budget Breakdown (Approximate):**
*   Morning Coffee: $6
*   Lunch: $15
*   Afternoon Coffee/Snack: $7
*   Concert Ticket (lowest price): $25
    **Total Estimated Cost: $53**

*To stay strictly under $50, consider opting for a simpler morning coffee, packing a snack, or finding a very inexpensive lunch option.*

Enjoy your spontaneous day trip through Berkeley!

--------------------------------------------------



In [ ]:
# --- Let's test the Day Trip Genie! ---

#Example:3

async def run_day_trip_genie():
    # Create a new, single-use session for this query
    day_trip_session = await session_service.create_session(
        app_name=day_trip_agent.name,
        user_id=my_user_id
    )

    # Note the new budget constraint in the query!
    query = "i wants go tiruvannamalai from mysore on 15th march i need clean and good plan "
    #query = "Plan a relaxing and artsy day trip near Sunnyvale, CA. Keep it affordable!"
    print(f"🗣️ User Query: '{query}'")

    await run_agent_query(day_trip_agent, query, day_trip_session, my_user_id)

await run_day_trip_genie()

🗣️ User Query: 'i wants go tiruvannamalai from mysore on 15th march i need clean and good plan '

🚀 Running query for agent: 'day_trip_agent' in session: 'ebe2b35b-46a2-4098-a7ff-f17d4578356f'...
EVENT: model_version='gemini-2.5-flash' content=Content(
  parts=[
    Part(
      text="""Here's a clean and good plan for a spontaneous day trip to Tiruvannamalai from Mysore on March 15th. This itinerary assumes travel by private car for maximum flexibility and to make the most of your day, given the significant travel time.

**Estimated Travel Time by Car:** Approximately 5-6 hours one-way (around 300-370 km).

---

### **Spontaneous Day Trip: Mysore to Tiruvannamalai (March 15th)**

**Mood:** Spiritual exploration, cultural immersion, serene experience.

**Budget:** Moderate (primarily for private car rental, food, and personal expenses). Entry to the main temple is free.

---

**🚗 Early Morning (4:30 AM - 10:30 AM): Journey to Tiruvannamalai**

*   **4:30 AM:** Depart from Mysore. It's r

Here's a clean and good plan for a spontaneous day trip to Tiruvannamalai from Mysore on March 15th. This itinerary assumes travel by private car for maximum flexibility and to make the most of your day, given the significant travel time.

**Estimated Travel Time by Car:** Approximately 5-6 hours one-way (around 300-370 km).

---

### **Spontaneous Day Trip: Mysore to Tiruvannamalai (March 15th)**

**Mood:** Spiritual exploration, cultural immersion, serene experience.

**Budget:** Moderate (primarily for private car rental, food, and personal expenses). Entry to the main temple is free.

---

**🚗 Early Morning (4:30 AM - 10:30 AM): Journey to Tiruvannamalai**

*   **4:30 AM:** Depart from Mysore. It's recommended to start very early to maximize your time in Tiruvannamalai. Pack some light snacks and water for the journey.
*   **Route:** You'll likely travel via Bangalore, then towards Vellore/Arcot and finally to Tiruvannamalai.
*   **Break:** A brief stop for tea/coffee and a quick stretch around the halfway mark.

**🧘‍♀️ Morning (10:30 AM - 1:30 PM): Divine Darshan at Arunachaleswarar Temple**

*   **10:30 AM:** Arrive in Tiruvannamalai. Head straight to the **Arunachaleswarar Temple**.
*   **10:30 AM - 1:00 PM:** Explore the vast and ancient Arunachaleswarar Temple. This temple is dedicated to Lord Shiva and is one of the Pancha Bhoota Sthalams, representing the element of fire (Agni). Marvel at its towering gopurams (temple towers), intricate sculptures, and the 1000-pillared hall. The temple is generally open for worship in the mornings from 5:30 AM to 12:30 PM.
*   **1:00 PM - 1:30 PM:** Take a moment for reflection or absorb the spiritual ambiance.

**🍽️ Lunch (1:30 PM - 2:30 PM): Local Flavors**

*   **1:30 PM - 2:30 PM:** Enjoy a traditional South Indian lunch at a local restaurant near the temple. There are several eateries offering authentic vegetarian meals.

**🙏 Afternoon (2:30 PM - 5:00 PM): Serenity at Ramana Ashram & Hill Exploration**

*   **2:30 PM - 4:00 PM:** Visit **Sri Ramana Ashramam**. This tranquil ashram was home to the renowned sage Sri Ramana Maharshi. It's a place for meditation, quiet contemplation, and learning about his teachings.
*   **4:00 PM - 5:00 PM:** If you're feeling adventurous and have sufficient time/energy, you can embark on a short trek up Arunachala Hill to visit the **Virupaksha Cave** or **Skandashramam**. These caves are significant for their association with Ramana Maharshi and offer panoramic views. Be mindful of the time for your return journey.

**🚗 Evening (5:00 PM - 11:00 PM): Return to Mysore**

*   **5:00 PM:** Begin your return journey to Mysore.
*   **Break:** A comfortable stop for tea/snacks and dinner on the way back.
*   **11:00 PM (approx.):** Arrive back in Mysore.

---

**Important Notes for your Trip:**

*   **Dress Code:** When visiting temples and ashrams, dress modestly. Traditional or formal clothing is generally recommended.
*   **Footwear:** You will need to remove your footwear before entering temples and ashrams.
*   **Photography:** Be aware of photography restrictions inside the main sanctums of temples.
*   **Hydration:** Carry enough water, especially if you plan to walk around the temple complex or trek.
*   **Traffic:** Factor in potential traffic, especially when passing through major towns or cities.
*   **Flexibility:** This is a suggested itinerary; feel free to adjust timings based on your preferences and how much time you wish to spend at each location.

--------------------------------------------------



---
## Part 2: Supercharging Agents with Custom Tools 🛠️

So far, we've used the powerful built-in `GoogleSearch` tool. But the true power of agents comes from connecting them to your own logic and data sources.

This is where **custom tools** come in. Let's explore three patterns for giving your agent new skills, using real-world, practical examples.

### 2.1 The Simple `FunctionTool`: Calling a Real-Time Weather API

The most direct way to create a tool is by writing a Python function. This is perfect for synchronous tasks like fetching data from an API.

**Key Concept:** The function's **docstring** is critical. The ADK uses it as the tool's official description, which the LLM reads to understand its purpose, parameters, and when to use it.

In this example, we'll create a tool that calls the **free, public U.S. National Weather Service API** to get a real-time forecast. No API key needed!

In [ ]:
"""# --- Tool Definition: A function that calls a live public API  (US)---
import requests
import json

# A simple lookup to avoid needing a separate geocoding API for this example
LOCATION_COORDINATES = {
    "sunnyvale": "37.3688,-122.0363",
    "san francisco": "37.7749,-122.4194",
    "lake tahoe": "39.0968,-120.0324",
    "mysore": "12.2958,76.6394",
    "bangalore": "12.9702,77.5603",
    "coorg": "12.4,75.7",
    "wayanad": "11.7032,76.0834",
    "ooty": "11.4134,76.6952",
    "nilakkal": "9.3815,76.9980",
    "sabarimala": "9.4346,77.0814",
    "pamba_dam": "9.3906,77.1598",
    "ranni": "9.3800,76.8100"

}

#Remove the commannting \/
def get_live_weather_forecast(location: str) -> dict:
    """"""Gets the current, real-time weather forecast for a specified location in the US.

    Args:
        location: The city name, e.g., "San Francisco".

    Returns:
        A dictionary containing the temperature and a detailed forecast.
    """"""
    print(f"🛠️ TOOL CALLED: get_live_weather_forecast(location='{location}')")

    # Find coordinates for the location
    normalized_location = location.lower()
    coords_str = None
    for key, val in LOCATION_COORDINATES.items():
        if key in normalized_location:
            coords_str = val
            break
    if not coords_str:
        return {"status": "error", "message": f"I don't have coordinates for {location}."}

    try:
        # NWS API requires 2 steps: 1. Get the forecast URL from the coordinates.
        points_url = f"https://api.weather.gov/points/{coords_str}"
        headers = {"User-Agent": "ADK Example Notebook"}
        points_response = requests.get(points_url, headers=headers)
        points_response.raise_for_status() # Raise an exception for bad status codes
        forecast_url = points_response.json()['properties']['forecast']

        # 2. Get the actual forecast from the URL.
        forecast_response = requests.get(forecast_url, headers=headers)
        forecast_response.raise_for_status()

        # Extract the relevant forecast details
        current_period = forecast_response.json()['properties']['periods'][0]
        return {
            "status": "success",
            "temperature": f"{current_period['temperature']}°{current_period['temperatureUnit']}",
            "forecast": current_period['detailedForecast']
        }
    except requests.exceptions.RequestException as e:
        return {"status": "error", "message": f"API request failed: {e}"}

# --- Agent Definition: An agent that USES the new tool ---

weather_agent = Agent(
    name="weather_aware_planner",
    model="gemini-2.5-flash",
    description="A trip planner that checks the real-time weather before making suggestions.",
    instruction="You are a cautious trip planner. Before suggesting any outdoor activities, you MUST use the `get_live_weather_forecast` tool to check conditions. Incorporate the live weather details into your recommendation.",
    tools=[get_live_weather_forecast]
)

print(f"🌦️ Agent '{weather_agent.name}' is created and can now call a live weather API!")"""

'# --- Tool Definition: A function that calls a live public API  (US)---\nimport requests\nimport json\n\n# A simple lookup to avoid needing a separate geocoding API for this example\nLOCATION_COORDINATES = {\n    "sunnyvale": "37.3688,-122.0363",\n    "san francisco": "37.7749,-122.4194",\n    "lake tahoe": "39.0968,-120.0324",\n    "mysore": "12.2958,76.6394",\n    "bangalore": "12.9702,77.5603",\n    "coorg": "12.4,75.7",\n    "wayanad": "11.7032,76.0834",\n    "ooty": "11.4134,76.6952",\n    "nilakkal": "9.3815,76.9980",\n    "sabarimala": "9.4346,77.0814",\n    "pamba_dam": "9.3906,77.1598",\n    "ranni": "9.3800,76.8100"\n\n}\n\ndef get_live_weather_forecast(location: str) -> dict:\n    Gets the current, real-time weather forecast for a specified location in the US.\n\n    Args:\n        location: The city name, e.g., "San Francisco".\n\n    Returns:\n        A dictionary containing the temperature and a detailed forecast.\n    \n    print(f"🛠️ TOOL CALLED: get_live_weather_forec

In [ ]:
# --- Tool Definition: A function that calls a live public API for (India) ---
import requests

# Predefined coordinates for Indian locations
LOCATION_COORDINATES = {
    "mysore": "12.2958,76.6394",
    "bangalore": "12.9702,77.5603",
    "coorg": "12.4,75.7",
    "wayanad": "11.7032,76.0834",
    "ooty": "11.4134,76.6952",
    "nilakkal": "9.3815,76.9980",
    "sabarimala": "9.4346,77.0814",
    "pamba_dam": "9.3906,77.1598",
    "ranni": "9.3800,76.8100"
}

# You need to sign up at OpenWeatherMap to get a free API key
OPENWEATHER_API_KEY = " "

def get_live_weather_forecast(location: str) -> dict:
    """Gets the current, real-time weather forecast for a specified location in India.

    Args:
        location: The city name, e.g., "Mysore".

    Returns:
        A dictionary containing the temperature and a detailed forecast.
    """
    print(f"🛠️ TOOL CALLED: get_live_weather_forecast(location='{location}')")

    # Find coordinates for the location
    normalized_location = location.lower()
    coords_str = None
    for key, val in LOCATION_COORDINATES.items():
        if key in normalized_location:
            coords_str = val
            break
    if not coords_str:
        return {"status": "error", "message": f"I don't have coordinates for {location}."}

    lat, lon = coords_str.split(",")

    try:
        # OpenWeatherMap API for current weather
        url = (
            f"https://api.openweathermap.org/data/2.5/weather?"
            f"lat={lat}&lon={lon}&appid={OPENWEATHER_API_KEY}&units=metric"
        )
        response = requests.get(url)
        response.raise_for_status()
        data = response.json()

        return {
            "status": "success",
            "temperature": f"{data['main']['temp']}°C",
            "weather": data['weather'][0]['description'],
            "humidity": f"{data['main']['humidity']}%",
            "wind_speed": f"{data['wind']['speed']} m/s"
        }

    except requests.exceptions.RequestException as e:
        return {"status": "error", "message": f"API request failed: {e}"}


# --- Agent Definition: An agent that USES the new tool ---
weather_agent = Agent(
    name="india_weather_planner",
    model="gemini-2.5-flash",
    description="A trip planner that checks the real-time weather in India before making suggestions.",
    instruction="You are a cautious trip planner. Before suggesting any outdoor activities, you MUST use the `get_live_weather_forecast` tool to check conditions. Incorporate the live weather details into your recommendation.",
    tools=[get_live_weather_forecast]
)

print(f"🌦️ Agent '{weather_agent.name}' is created and can now call a live weather API for India!")

🌦️ Agent 'india_weather_planner' is created and can now call a live weather API for India!


In [ ]:
# --- Let's test the Weather-Aware Planner ---

async def run_weather_planner_test():
    weather_session = await session_service.create_session(app_name=weather_agent.name, user_id=my_user_id)
    #query = "I want to go hiking near Kukkarahalli Lake(mysore), what's the weather like?"
    #query = "I want to go to play cricket at 5pm on railway ground (mysore), what's the weather like?"
    query ="what is temp in mysore now"
    print(f"🗣️ User Query: '{query}'")
    await run_agent_query(weather_agent, query, weather_session, my_user_id)

await run_weather_planner_test()

🗣️ User Query: 'what is temp in mysore now'

🚀 Running query for agent: 'india_weather_planner' in session: '904f49ae-fcf3-40ab-8cd0-95ea55f972e7'...


EVENT: model_version='gemini-2.5-flash' content=Content(
  parts=[
    Part(
      function_call=FunctionCall(
        args={
          'location': 'Mysore'
        },
        id='adk-1934a1d4-c833-423b-a38b-f1bfb357590d',
        name='get_live_weather_forecast'
      ),
      thought_signature=b"\n\xbe\x02\x01\xbe>\xf6\xfb\xcc\x16\xfd/\x86\x9f\n\x02\xfa\x8eL2\x04\xfe\xac'2\x96\xd5TQ_\x92\x1eT\x14\x8b\x1e\xf7b\xa5\x13v\xa5\x07!\x99D\rI=FU\xc0q}\xc4a\xf0\x19\x93\xf9%\x99\r\xd8\x1bt\xae\xe86\x81v\x9eV\xe9O\xe2(\xa3.\xaa\xce\xc2\xab\x80\xbe\x11\x0f\xc1\x9e\x827\xd0\xc9\x97\xa5\xb78...'
    ),
  ],
  role='model'
) grounding_metadata=None partial=None turn_complete=None finish_reason=<FinishReason.STOP: 'STOP'> error_code=None error_message=None interrupted=None custom_metadata=None usage_metadata=GenerateContentResponseUsageMetadata(
  candidates_token_count=20,
  prompt_token_count=179,
  prompt_tokens_details=[
    ModalityTokenCount(
      modality=<MediaModality.TEXT: 'TEXT'>,
      

The current temperature in Mysore is 33.62°C with a clear sky and a wind speed of 4.23 m/s. The humidity is 18%.

--------------------------------------------------



## 2.2 The Agent-as-a-Tool: Consulting a Specialist 🧑‍🍳

Why build one agent that does everything when you can build a **team of specialist agents?** The **Agent-as-a-Tool** pattern allows one agent to delegate a task to another agent.

**Key Concept:** This is different from a sub-agent. When Agent A calls Agent B as a tool, Agent B's response is passed **back to Agent A**. Agent A then uses that information to form its own final response to the user. It's a powerful way to compose complex behaviors from simpler, focused, and reusable agents.

### How It Works

Our top-level agent, the `trip_data_concierge_agent`, acts as the **Orchestrator**. It has two tools at its disposal:

1.  `call_db_agent`: A function that internally calls our `db_agent` to fetch raw data.
2.  `call_concierge_agent`: A function that calls the `concierge_agent`.

The `concierge_agent` itself has a tool: the `food_critic_agent`.

The flow for a complex query is:

1.  **User** asks the `trip_data_concierge_agent` for a hotel and a nearby restaurant.
2.  The **Orchestrator** first calls `call_db_agent` to get hotel data.
3.  The data is saved in `tool_context.state`.
4.  The **Orchestrator** then calls `call_concierge_agent`, which retrieves the hotel data from the context.
5.  The `concierge_agent` receives the request and decides it needs to use its own tool, the `food_critic_agent`.
6.  The `food_critic_agent` provides a witty recommendation.
7.  The `concierge_agent` gets the critic's response and politely formats it.
8.  This final, polished response is returned to the **Orchestrator**, which presents it to the user.

                         +-----------------------------------------------------------+
                         |              🧭 Trip Data Concierge Agent                 |
                         |-----------------------------------------------------------|
                         |  Model: gemini-2.5-flash                                  |
                         |  Description:                                             |
                         |   Orchestrates database query and travel recommendation  |
                         |-----------------------------------------------------------|
                         |  🔧 Tools:                                                |
                         |   1. call_db_agent                                        |
                         |   2. call_concierge_agent                                 |
                         +-----------------------------------------------------------+
                                      /                                \
                                     /                                  \
                                    ▼                                    ▼
        +-------------------------------------------+    +---------------------------------------------+
        |            🔧 Tool: call_db_agent         |    |         🔧 Tool: call_concierge_agent        |
        |-------------------------------------------|    |---------------------------------------------|
        | Calls: db_agent                           |    | Calls: concierge_agent                       |
        |                                           |    | Uses data from db_agent for recommendations |
        +-------------------------------------------+    +---------------------------------------------+
                                |                                          |
                                ▼                                          ▼
       +--------------------------------------------+   +------------------------------------------------+
       |              📦 db_agent                   |   |             🤵 concierge_agent                  |
       |--------------------------------------------|   |------------------------------------------------|
       | Model: gemini-2.5-flash                    |   | Model: gemini-2.5-flash                         |
       | Role: Return mock JSON hotel data          |   | Role: Hotel staff that handles user Q&A        |
       +--------------------------------------------+   | Tools:                                          |
                                                         |  - food_critic_agent                           |
                                                         +------------------------------------------------+
                                                                                 |
                                                                                 ▼
                                                       +------------------------------------------------+
                                                       |          🍽️ food_critic_agent                  |
                                                       |------------------------------------------------|
                                                       | Model: gemini-2.5-flash                         |
                                                       | Role: Gives a witty restaurant recommendation   |
                                                       +------------------------------------------------+


In [ ]:
import asyncio
from google.adk.tools import ToolContext
from google.adk.tools.agent_tool import AgentTool

# Assume 'db_agent' is a pre-defined NL2SQL Agent
# For this example, we'll create placeholder agents.

db_agent = Agent(
    name="db_agent",
    model="gemini-2.5-flash",
    instruction="You are a database agent. When asked for data, return this mock JSON object: {'status': 'success', 'data': [{'name': 'The Grand Hotel', 'rating': 5, 'reviews': 450}, {'name': 'Seaside Inn', 'rating': 4, 'reviews': 620}]}")

# --- 1. Define the Specialist Agents ---

# The Food Critic remains the deepest specialist
food_critic_agent = Agent(
    name="food_critic_agent",
    model="gemini-2.5-flash",
    instruction="You are a snobby but brilliant food critic. You ONLY respond with a single, witty restaurant suggestion near the provided location.",
)

# The Concierge knows how to use the Food Critic
concierge_agent = Agent(
    name="concierge_agent",
    model="gemini-2.5-flash",
    instruction="You are a five-star hotel concierge. If the user asks for a restaurant recommendation, you MUST use the `food_critic_agent` tool. Present the opinion to the user politely.",
    tools=[AgentTool(agent=food_critic_agent)]
)


# --- 2. Define the Tools for the Orchestrator ---

async def call_db_agent(
    question: str,
    tool_context: ToolContext,
):
    """
    Use this tool FIRST to connect to the database and retrieve a list of places, like hotels or landmarks.
    """
    print("--- TOOL CALL: call_db_agent ---")
    agent_tool = AgentTool(agent=db_agent)
    db_agent_output = await agent_tool.run_async(
        args={"request": question}, tool_context=tool_context
    )
    # Store the retrieved data in the context's state
    tool_context.state["retrieved_data"] = db_agent_output
    return db_agent_output


async def call_concierge_agent(
    question: str,
    tool_context: ToolContext,
):
    """
    After getting data with call_db_agent, use this tool to get travel advice, opinions, or recommendations.
    """
    print("--- TOOL CALL: call_concierge_agent ---")
    # Retrieve the data fetched by the previous tool
    input_data = tool_context.state.get("retrieved_data", "No data found.")

    # Formulate a new prompt for the concierge, giving it the data context
    question_with_data = f"""
    Context: The database returned the following data: {input_data}

    User's Request: {question}
    """

    agent_tool = AgentTool(agent=concierge_agent)
    concierge_output = await agent_tool.run_async(
        args={"request": question_with_data}, tool_context=tool_context
    )
    return concierge_output


# --- 3. Define the Top-Level Orchestrator Agent ---

trip_data_concierge_agent = Agent(
    name="trip_data_concierge",
    model="gemini-2.5-flash",
    description="Top-level agent that queries a database for travel data, then calls a concierge agent for recommendations.",
    tools=[call_db_agent, call_concierge_agent],
    instruction="""
    You are a master travel planner who uses data to make recommendations.

    1.  **ALWAYS start with the `call_db_agent` tool** to fetch a list of places (like hotels) that match the user's criteria.

    2.  After you have the data, **use the `call_concierge_agent` tool** to answer any follow-up questions for recommendations, opinions, or advice related to the data you just found.
    """,
)

print(f"✅ Orchestrator Agent '{trip_data_concierge_agent.name}' is defined and ready.")

✅ Orchestrator Agent 'trip_data_concierge' is defined and ready.


In [ ]:
# --- Let's test the Trip Data Concierge Agent ---

async def run_trip_data_concierge():
    """
    Sets up a session and runs a query against the top-level
    trip_data_concierge_agent.
    """
    # Create a new, single-use session for this query
    concierge_session = await session_service.create_session(
        app_name=trip_data_concierge_agent.name,
        user_id=my_user_id
    )

    # This query is specifically designed to trigger the full two-step process:
    # 1. Get data from the db_agent.
    # 2. Get a recommendation from the concierge_agent based on that data.
    #query = "Find the top-rated hotels in San Francisco from the database, then suggest a dinner spot near the one with the most reviews."
    query="find the good hotels in mysore"
    print(f"🗣️ User Query: '{query}'")

    # We call our existing helper function with the top-level orchestrator agent
    await run_agent_query(trip_data_concierge_agent, query, concierge_session, my_user_id)

# Run the test
await run_trip_data_concierge()

🗣️ User Query: 'find the good hotels in mysore'

🚀 Running query for agent: 'trip_data_concierge' in session: 'a562297e-7ec5-43bb-9dc9-9854db86e52d'...
EVENT: model_version='gemini-2.5-flash' content=Content(
  parts=[
    Part(
      function_call=FunctionCall(
        args={
          'question': 'good hotels in Mysore'
        },
        id='adk-54145131-18b8-4280-9f8f-e79835ab0f7d',
        name='call_db_agent'
      ),
      thought_signature=b'\n\xba\x02\x01\xbe>\xf6\xfbu\xa5\x13\x7f.h\x89|;]\xf5\xc1\xf3\xae\x86\xeb%\x96\xe5dg:\xec\x85\r\x94\x9dZ\xa3\x0e\x14\xd2$\xf7\x9d\xb1\x17\xb5\xbd\xc1t0g\xf1\xe9Z\ts6-YJ\xcbi\xda\xcb9;]U\x8f\x15$\x8aB\xaa\x8d\x9ah\xe6EgYWX\x92\x18\x198\xa1\x94\xe1\x07\xdd\xa1w\xf4\x90\\...'
    ),
  ],
  role='model'
) grounding_metadata=None partial=None turn_complete=None finish_reason=<FinishReason.STOP: 'STOP'> error_code=None error_message=None interrupted=None custom_metadata=None usage_metadata=GenerateContentResponseUsageMetadata(
  candidates_token_

Based on the provided data, "The Grand Hotel" stands out with a 5-star rating and 450 reviews. "Seaside Inn" is also a good option with a 4-star rating and 620 reviews.

--------------------------------------------------



---
## Part 3: Agent with a Memory - The Adaptive Planner 🗺️

Now, let's see an agent that not only **remembers** but also **adapts**. We'll challenge the `multi_day_trip_agent` to re-plan part of its itinerary based on our feedback. This is a much more realistic test of conversational AI.

```
+-----------------------------------------------------+
|         Adaptive Multi-Day Trip Agent 🗺️           |
|-----------------------------------------------------|
|  Model: gemini-2.5-flash                            |
|  Description:                                       |
|   Builds multi-day travel itineraries step-by-step, |
|   remembers previous days, adapts to feedback       |
|-----------------------------------------------------|
|  🔧 Tools:                                          |
|   - Google Search                                   |
|-----------------------------------------------------|
|  🧠 Capabilities:                                   |
|   - Memory of past conversation & preferences       |
|   - Progressive planning (1 day at a time)          |
|   - Adapts to user feedback                         |
|   - Ensures activity variety across days            |
+-----------------------------------------------------+

            ▲
            |
    +---------------------------+
    |     User Interaction      |
    |---------------------------|
    | - Destination             |
    | - Trip duration           |
    | - Interests & feedback    |
    +---------------------------+

            |
            ▼

+-----------------------------------------------------+
|        Day-by-Day Itinerary Generation              |
|-----------------------------------------------------|
|  🗓️ Day N Output (Markdown format):                 |
|   - Morning / Afternoon / Evening activities        |
|   - Personalized & context-aware                    |
|   - Changes accepted, feedback acknowledged         |
+-----------------------------------------------------+

            |
            ▼

+-----------------------------------------------------+
|        Next Day Planning Triggered 🚀               |
|-----------------------------------------------------|
| - Builds on prior days                              |
| - Avoids repetition                                 |
| - Asks user for confirmation before proceeding      |
+-----------------------------------------------------+
```


In [ ]:
# --- Agent Definition: The Adaptive Planner ---

def create_multi_day_trip_agent():
    """Create the Progressive Multi-Day Trip Planner agent"""
    return Agent(
        name="multi_day_trip_agent",
        model="gemini-2.5-flash",
        description="Agent that progressively plans a multi-day trip, remembering previous days and adapting to user feedback.",
        instruction="""
        You are the "Adaptive Trip Planner" 🗺️ - an AI assistant that builds multi-day travel itineraries step-by-step.

        Your Defining Feature:
        You have short-term memory. You MUST refer back to our conversation to understand the trip's context, what has already been planned, and the user's preferences. If the user asks for a change, you must adapt the plan while keeping the unchanged parts consistent.

        Your Mission:
        1.  **Initiate**: Start by asking for the destination, trip duration, and interests.
        2.  **Plan Progressively**: Plan ONLY ONE DAY at a time. After presenting a plan, ask for confirmation.
        3.  **Handle Feedback**: If a user dislikes a suggestion (e.g., "I don't like museums"), acknowledge their feedback, and provide a *new, alternative* suggestion for that time slot that still fits the overall theme.
        4.  **Maintain Context**: For each new day, ensure the activities are unique and build logically on the previous days. Do not suggest the same things repeatedly.
        5.  **Final Output**: Return each day's itinerary in MARKDOWN format.
        """,
        tools=[google_search]
    )

multi_day_agent = create_multi_day_trip_agent()
print(f"🗺️ Agent '{multi_day_agent.name}' is created and ready to plan and adapt!")

🗺️ Agent 'multi_day_trip_agent' is created and ready to plan and adapt!


### Scenario 3a: Agent WITH Memory (Using a SINGLE Session) ✅

First, let's see the correct way to do it. We will use the **exact same `trip_session` object** for the entire conversation. Watch how the agent remembers the context from Turn 1 to correctly handle the requests in Turn 2 and 3.

In [ ]:
# --- Scenario 2: Testing Adaptation and Memory ---

async def run_adaptive_memory_demonstration():
    print("### 🧠 DEMO 2: AGENT THAT ADAPTS (SAME SESSION) ###")

    # Create ONE session that we will reuse for the whole conversation
    trip_session = await session_service.create_session(
        app_name=multi_day_agent.name,
        user_id=my_user_id
    )
    print(f"Created a single session for our trip: {trip_session.id}")

    # --- Turn 1: The user initiates the trip ---
    #query1 = "Hi! I want to plan a 2-day trip to Lisbon, Portugal. I'm interested in historic sites and great local food."
    query1="Hi! I want to plan a 2-day trip to coorg, I'm interested in historic sites and great local food."
    print(f"\n🗣️ User (Turn 1): '{query1}'")
    await run_agent_query(multi_day_agent, query1, trip_session, my_user_id)

    # --- Turn 2: The user gives FEEDBACK and asks for a CHANGE ---
    # We use the EXACT SAME `trip_session` object!
    query2 = "That sounds pretty good, but I'm not a huge fan of castles. Can you replace the morning activity for Day 1 with something else historical?"
    print(f"\n🗣️ User (Turn 2 - Feedback): '{query2}'")
    await run_agent_query(multi_day_agent, query2, trip_session, my_user_id)

    # --- Turn 3: The user confirms and asks to continue ---
    query3 = "Yes, the new plan for Day 1 is perfect! Please plan Day 2 now, keeping the food theme in mind."
    print(f"\n🗣️ User (Turn 3 - Confirmation): '{query3}'")
    await run_agent_query(multi_day_agent, query3, trip_session, my_user_id)

await run_adaptive_memory_demonstration()

### 🧠 DEMO 2: AGENT THAT ADAPTS (SAME SESSION) ###
Created a single session for our trip: 2813c809-a110-471a-8a81-7272a8de35a6

🗣️ User (Turn 1): 'Hi! I want to plan a 2-day trip to coorg, I'm interested in historic sites and great local food.'

🚀 Running query for agent: 'multi_day_trip_agent' in session: '2813c809-a110-471a-8a81-7272a8de35a6'...
EVENT: model_version='gemini-2.5-flash' content=Content(
  parts=[
    Part(
      text="""Hello! I can definitely help you plan a fantastic 2-day trip to Coorg, focusing on historic sites and delicious local food. Coorg, also known as Kodagu, is a beautiful hill station with a rich history and unique cuisine.

Let's start by planning Day 1.

Here's a possible itinerary for your first day:

**Day 1: Historical Immersion & Kodava Flavors**

*   **Morning (9:00 AM - 1:00 PM): Madikeri Fort & Museum**
    Begin your historical journey at the Madikeri Fort, a prominent 17th-century landmark built by Muddu Raja. It was later modified by Tipu Sulta

Hello! I can definitely help you plan a fantastic 2-day trip to Coorg, focusing on historic sites and delicious local food. Coorg, also known as Kodagu, is a beautiful hill station with a rich history and unique cuisine.

Let's start by planning Day 1.

Here's a possible itinerary for your first day:

**Day 1: Historical Immersion & Kodava Flavors**

*   **Morning (9:00 AM - 1:00 PM): Madikeri Fort & Museum**
    Begin your historical journey at the Madikeri Fort, a prominent 17th-century landmark built by Muddu Raja. It was later modified by Tipu Sultan and the British. Inside the fort, you'll find a museum showcasing artifacts and photographs from Coorg's past, and you can also enjoy panoramic views of the city. The fort also houses a temple, a church, and a library.

*   **Lunch (1:00 PM - 2:30 PM): Authentic Kodava Lunch**
    Head to a local restaurant for an authentic Kodava lunch. Coorg cuisine is known for its rich flavors, often using locally grown spices. You could try "Pandi Curry" (pork curry), a highly emblematic dish, or "Kadambuttu" (steamed rice dumplings) which are often paired with curries. "Akki Otti" (rice flatbread) is another staple. Restaurants like "Coorg Cuisine" or "Taste of Coorg" are known for traditional Kodava dishes.

*   **Afternoon (2:30 PM - 4:30 PM): Omkareshwara Temple**
    Visit the Omkareshwara Temple, an ancient temple built in 1820 by Lingarajendra II. This temple is unique for its blend of Islamic and Gothic architectural styles and is dedicated to Lord Shiva.

*   **Late Afternoon (4:30 PM - 6:00 PM): Raja's Tomb (Gaddige)**
    Explore Gaddige, also known as Raja's Tomb, the final resting place of Kodava royalty, including Virarajendra and Lingarajendra. These tombs showcase a blend of Indo-Sarcenic and Muhammadan architectural styles and are surrounded by a beautifully landscaped garden.

*   **Evening (6:00 PM onwards): Raja's Seat**
    Conclude your day with a visit to Raja's Seat. This garden offers breathtaking panoramic views of the surrounding hills and valleys, and it was a favored leisure spot for the kings of Kodagu. It's an excellent place to witness a spectacular sunset.

How does this sound for Day 1? Would you like any adjustments, or are you ready to move on to Day 2?

--------------------------------------------------


🗣️ User (Turn 2 - Feedback): 'That sounds pretty good, but I'm not a huge fan of castles. Can you replace the morning activity for Day 1 with something else historical?'

🚀 Running query for agent: 'multi_day_trip_agent' in session: '2813c809-a110-471a-8a81-7272a8de35a6'...
EVENT: model_version='gemini-2.5-flash' content=Content(
  parts=[
    Part(
      text="""Understood! You're looking for another historical site that isn't a fort or castle. Coorg has a rich history beyond just its forts.

Here's an updated plan for Day 1, replacing the Madikeri Fort with a different significant historical and spiritual site:

**Day 1: Historical Immersion & Kodava Flavors (Revised Morning)**

*   **Morning (9:00 AM - 1:00 PM): Talakaveri**
    Instead of the fort, we'll head to Talakaveri, the sacred source of the River Cauvery, nestled in the Brahmagiri Hills. This is a highly significant pilgrimage site and offers breathtaking views of the sur

Understood! You're looking for another historical site that isn't a fort or castle. Coorg has a rich history beyond just its forts.

Here's an updated plan for Day 1, replacing the Madikeri Fort with a different significant historical and spiritual site:

**Day 1: Historical Immersion & Kodava Flavors (Revised Morning)**

*   **Morning (9:00 AM - 1:00 PM): Talakaveri**
    Instead of the fort, we'll head to Talakaveri, the sacred source of the River Cauvery, nestled in the Brahmagiri Hills. This is a highly significant pilgrimage site and offers breathtaking views of the surrounding Western Ghats. Historically and spiritually, the river is central to the culture and life of the region. There's a small temple dedicated to Goddess Kaveri and a tank where devotees take holy dips. You can also climb to the top of the Brahmagiri Peak for even more expansive panoramic views. It's about an an hour's drive from Madikeri.

*   **Lunch (1:00 PM - 2:30 PM): Authentic Kodava Lunch**
    After visiting Talakaveri, we'll head back towards Madikeri for an authentic Kodava lunch. You could try "Pandi Curry" (pork curry), "Kadambuttu" (steamed rice dumplings), or "Akki Otti" (rice flatbread). Restaurants like "Coorg Cuisine" or "Taste of Coorg" are known for traditional Kodava dishes.

*   **Afternoon (2:30 PM - 4:30 PM): Omkareshwara Temple**
    Visit the Omkareshwara Temple, an ancient temple built in 1820 by Lingarajendra II. This temple is unique for its blend of Islamic and Gothic architectural styles and is dedicated to Lord Shiva.

*   **Late Afternoon (4:30 PM - 6:00 PM): Raja's Tomb (Gaddige)**
    Explore Gaddige, also known as Raja's Tomb, the final resting place of Kodava royalty, including Virarajendra and Lingarajendra. These tombs showcase a blend of Indo-Sarcenic and Muhammadan architectural styles and are surrounded by a beautifully landscaped garden.

*   **Evening (6:00 PM onwards): Raja's Seat**
    Conclude your day with a visit to Raja's Seat. This garden offers breathtaking panoramic views of the surrounding hills and valleys, and it was a favored leisure spot for the kings of Kodagu. It's an excellent place to witness a spectacular sunset.

How does this revised Day 1 plan sound to you, particularly with Talakaveri as the morning activity?

--------------------------------------------------


🗣️ User (Turn 3 - Confirmation): 'Yes, the new plan for Day 1 is perfect! Please plan Day 2 now, keeping the food theme in mind.'

🚀 Running query for agent: 'multi_day_trip_agent' in session: '2813c809-a110-471a-8a81-7272a8de35a6'...

--------------------------------------------------
✅ Final Response:


An error occurred: 
On how to mitigate this issue, please refer to:

https://google.github.io/adk-docs/agents/models/#error-code-429-resource_exhausted


429 RESOURCE_EXHAUSTED. {'error': {'code': 429, 'message': 'You exceeded your current quota, please check your plan and billing details. For more information on this error, head to: https://ai.google.dev/gemini-api/docs/rate-limits. To monitor your current usage, head to: https://ai.dev/rate-limit. \n* Quota exceeded for metric: generativelanguage.googleapis.com/generate_content_free_tier_requests, limit: 20, model: gemini-2.5-flash\nPlease retry in 31.718887679s.', 'status': 'RESOURCE_EXHAUSTED', 'details': [{'@type': 'type.googleapis.com/google.rpc.Help', 'links': [{'description': 'Learn more about Gemini API quotas', 'url': 'https://ai.google.dev/gemini-api/docs/rate-limits'}]}, {'@type': 'type.googleapis.com/google.rpc.QuotaFailure', 'violations': [{'quotaMetric': 'generativelanguage.googleapis.com/generate_content_free_tier_requests', 'quotaId': 'GenerateRequestsPerDayPerProjectPerModel-FreeTier', 'quotaDimensions': {'location': 'global', 'model': 'gemini-2.5-flash'}, 'quotaValue': '20'}]}, {'@type': 'type.googleapis.com/google.rpc.RetryInfo', 'retryDelay': '31s'}]}}

--------------------------------------------------



### Scenario 3b: Agent WITHOUT Memory (Using SEPARATE Sessions) ❌

Now, let's see what happens if we mess up our session management. Here, we'll give the agent a case of amnesia by creating a **brand new, separate session for each turn**.

Pay close attention to the agent's response to the second query. Because it's in a new session, it has no memory of the trip to Lisbon we just discussed!

In [ ]:
# --- Scenario 2b: Demonstrating Memory FAILURE ---

async def run_memory_failure_demonstration():
    print("\n" + "#"*60)
    print("### 🧠 DEMO 2b: AGENT WITH AMNESIA (SEPARATE SESSIONS) ###")
    print("#"*60)

    # --- Turn 1: The user initiates the trip in the FIRST session ---
    #query1 = "Hi! I want to plan a 2-day trip to Lisbon, Portugal. I'm interested in historic sites and great local food."
    query1="Hi! I want to plan a day trip to mysore, I'm interested great local food."
    session_one = await session_service.create_session(
        app_name=multi_day_agent.name,
        user_id=my_user_id
    )
    print(f"\nCreated a session for Turn 1: {session_one.id}")
    print(f"🗣️ User (Turn 1): '{query1}'")
    await run_agent_query(multi_day_agent, query1, session_one, my_user_id)

    # --- Turn 2: The user asks to continue... but in a completely NEW session ---
    query2 ="" #"Yes, that looks perfect! Please plan Day 2."
    session_two = await session_service.create_session(
        app_name=multi_day_agent.name,
        user_id=my_user_id
    )
    print(f"\nCreated a BRAND NEW session for Turn 2: {session_two.id}")
    print(f"🗣️ User (Turn 2): '{query2}'")
    await run_agent_query(multi_day_agent, query2, session_two, my_user_id)

await run_memory_failure_demonstration()


############################################################
### 🧠 DEMO 2b: AGENT WITH AMNESIA (SEPARATE SESSIONS) ###
############################################################

Created a session for Turn 1: b126982f-8e08-4c21-87f5-40036e9badc9
🗣️ User (Turn 1): 'Hi! I want to plan a day trip to mysore, I'm interested great local food.'

🚀 Running query for agent: 'multi_day_trip_agent' in session: 'b126982f-8e08-4c21-87f5-40036e9badc9'...
EVENT: model_version='gemini-2.5-flash' content=Content(
  parts=[
    Part(
      text="""Sounds like a delicious plan! Mysore is a fantastic choice for a day trip, especially if you're keen on local food.

Here's a possible itinerary for your day trip to Mysore, focusing on its culinary delights and a couple of key sights:

---

### **Mysore Food & Culture Day Trip**

*   **Morning (9:00 AM - 1:00 PM): Royal Heritage & Breakfast Bliss**
    *   **9:00 AM - 10:00 AM: Breakfast at Mylari Hotel (Original)** Kickstart your day with Mysore's legenda

Sounds like a delicious plan! Mysore is a fantastic choice for a day trip, especially if you're keen on local food.

Here's a possible itinerary for your day trip to Mysore, focusing on its culinary delights and a couple of key sights:

---

### **Mysore Food & Culture Day Trip**

*   **Morning (9:00 AM - 1:00 PM): Royal Heritage & Breakfast Bliss**
    *   **9:00 AM - 10:00 AM: Breakfast at Mylari Hotel (Original)** Kickstart your day with Mysore's legendary Benne Dosa (butter dosa) at the iconic Mylari Hotel. Be prepared for a wait, but it's worth it!
    *   **10:30 AM - 1:00 PM: Mysore Palace Exploration** After breakfast, head straight to the magnificent Mysore Palace. Spend a good couple of hours marveling at its architecture, intricate designs, and rich history.

*   **Lunch (1:00 PM - 2:30 PM): Traditional Thali Experience**
    *   **1:00 PM - 2:30 PM: Traditional South Indian Lunch** Enjoy a traditional South Indian thali at a local restaurant like Hotel RRR (known for non-vegetarian options if you're interested) or Vinayaka Mylari (another branch with a broader menu than the breakfast spot) for authentic vegetarian fare.

*   **Afternoon (2:30 PM - 6:00 PM): Sweet Treats, Local Markets & Views**
    *   **2:30 PM - 3:30 PM: Mysore Silk Saree & Sandalwood Shopping (Optional)** Explore the bustling KR Market for local spices, flowers, and a glimpse into daily life. If interested, you can also visit a government emporium for authentic Mysore Silk Sarees or sandalwood products.
    *   **3:30 PM - 4:30 PM: Mysore Pak Indulgence** No trip to Mysore is complete without tasting its signature sweet, Mysore Pak. Head to Guru Sweet Mart or Mahalakshmi Sweets for some of the best in town.
    *   **4:30 PM - 6:00 PM: Chamundi Hills & Nandi Statue** Drive up to Chamundi Hills to visit the Chamundeshwari Temple and enjoy panoramic views of Mysore city. Don't miss the colossal Nandi Bull statue on the way down.

*   **Evening (6:00 PM onwards): Evening Snacks & Departure**
    *   **6:00 PM - 7:00 PM: Evening Snacks/Coffee** Before heading back, grab some filter coffee and a light snack like Maddur Vada at a local cafe.

---

How does this sound for your day trip? We can adjust anything you like!

--------------------------------------------------


Created a BRAND NEW session for Turn 2: dc4923a5-c400-4482-aa0a-46b972c6713b
🗣️ User (Turn 2): ''

🚀 Running query for agent: 'multi_day_trip_agent' in session: 'dc4923a5-c400-4482-aa0a-46b972c6713b'...
EVENT: model_version='gemini-2.5-flash' content=Content(
  parts=[
    Part(
      text="""Hello! I'm your Adaptive Trip Planner. To help me create the perfect itinerary for you, please tell me:

1.  **What is your desired destination?**
2.  **How many days will your trip be?**
3.  **What are your interests or what kind of activities do you enjoy?** (e.g., history, food, outdoors, relaxation, adventure, art, nightlife)"""
    ),
  ],
  role='model'
) grounding_metadata=GroundingMetadata() partial=None turn_complete=None finish_reason=<FinishReason.STOP: 'STOP'> error_code=None error_message=None interrupted=None custom_metadata=None usage_metadata=GenerateContentResponseUsageMetadata(
  candidates_token_count=88,
  prompt_token_count=

Hello! I'm your Adaptive Trip Planner. To help me create the perfect itinerary for you, please tell me:

1.  **What is your desired destination?**
2.  **How many days will your trip be?**
3.  **What are your interests or what kind of activities do you enjoy?** (e.g., history, food, outdoors, relaxation, adventure, art, nightlife)

--------------------------------------------------



See? The agent was confused! It likely asked what destination or what trip we were talking about. Because the second query was in a fresh, isolated session, the agent had no memory of planning Day 1 in Lisbon.

This perfectly illustrates why **managing sessions is the key to building truly conversational agents!**

---
## 🎉 Congratulations! 🎉

Congratulations on completing your ADK adventure into Tools and Memory! You've taken a massive leap from building single-shot agents to creating dynamic, stateful AI systems.

Let's recap the powerful concepts you've mastered:

- **Fundamental Agent & Tools**: You started by building a "Day Trip Genie" and equipped it with its first tool, GoogleSearch.

- **Custom Function Tools**: You gave your agent a new sense by creating a custom tool to fetch live data from the U.S. National Weather Service API.

- **Agent-as-a-Tool**: You orchestrated a sophisticated hierarchy where agents delegate tasks to other, more specialized agents, creating a collaborative team.

- **The Power of Memory**: Most importantly, you saw firsthand how managing a single, persistent Session allows an agent to remember context, adapt to user feedback, and conduct a meaningful, multi-turn conversation.

```
   __            /\_/\         /\_/\        /\_/\         __             (\__/)
o-''|\_____/).  ( o.o )       ( -.- )      ( ^_^ )     o-''|\_____/).    ( ^_^ )
 \_/|_)     )    > ^ <         > * <        >💖<         \_/|_)     )     / >🌸< \
    \  __  /                                              \  __  /         /   \
    (_/ (_/                                               (_/ (_/        (___|___)
```
